<a href="https://colab.research.google.com/github/Yangel300/BreathIA/blob/AUDIO_DIV/Ventanas_audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformación del dataset (audio) en ventanas de n segundos con label

### Instalar e importar librerías necesarias

In [ ]:
!pip install selenium
!pip install pdfrw
!pip install kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.7/512.7 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.5/69.5 kB 2.5 MB/s eta 0:00:00


In [ ]:
import librosa
import IPython.display as ipd
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Audio
import soundfile as sf
from numpy import pi
from scipy.fftpack import fft, fftfreq
import os
import librosa
import torch

### Importar dataset (cuando no está)



In [ ]:
from google.colab import files

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Descargar el dataset.
!kaggle datasets download -d vbookshelf/respiratory-sound-database

Dataset URL: https://www.kaggle.com/datasets/vbookshelf/respiratory-sound-database
License(s): unknown
100% 3.67G/3.69G [00:46<00:00, 115MB/s] 
100% 3.69G/3.69G [00:46<00:00, 85.9MB/s]


In [ ]:
!unzip respiratory-sound-database.zip

Archive:  respiratory-sound-database.zip
  inflating: Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/101_1b1_Al_sc_Meditron.txt  
  inflating: Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/101_1b1_Al_sc_Meditron.wav  
  inflating: Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/101_1b1_Pr_sc_Meditron.txt  
  inflating: Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/101_1b1_Pr_sc_Meditron.wav  
  inflating: Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/102_1b1_Ar_sc_Meditron.txt  
  inflating: Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/102_1b1_Ar_sc_Meditron.wav  
  inflating: Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/103_2b2_Ar_mc_LittC2SE.txt  
  inflating: Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/103_2b2_Ar_mc_LittC2SE.wav  
  inflating: Respiratory_Sound_

## Procedimiento para un único archivo

### Obtener los audios

In [ ]:
audio_dir = '/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/'
audio_data_list = []

for filename in os.listdir(audio_dir):
    if filename.endswith('.wav'):
        file_path = os.path.join(audio_dir, filename)
        y, sr = librosa.load(file_path, sr=None) # Cargar los audios con librosa
        audio_data_list.append(y)

print(f"Carga de {len(audio_data_list)} archivos de audio.")

Carga de 920 archivos de audio.


In [ ]:
# Leer un audio para hacerle las pruebas

audio_file_path = None
for filename in os.listdir(audio_dir):
    if filename.endswith('.wav'):
        audio_file_path = os.path.join(audio_dir, filename)
        break

if audio_file_path:
    y, sr = librosa.load(audio_file_path) # Se define el audio como una señal de onda y sr referente al sample rate
    print(f"Successfully loaded audio file: {audio_file_path}")
else:
    print("No .wav files found in the directory.")

Successfully loaded audio file: /content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/145_2b2_Lr_mc_AKGC417L.wav


In [ ]:
Audio(y, rate=sr)
print(Audio(y,rate=sr))

<IPython.lib.display.Audio object>


In [ ]:
#y, sr = librosa.load("/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/145_2b2_Lr_mc_AKGC417L.wav")

print("Audio:")
display(Audio(y, rate=sr))

Audio:


In [32]:
#Estudiar cada uno de los parámetros del objeto creado para leer el archivo de audio.
F= sr #sample rate o frecuencia del audio
dt = y.size / sr  #tiempo x muestra
T = 1 / sr #Período
t = np.r_[0:dt:T]
print(
    f'y[t] tiene {y.size} muestras', #Cantidad de muestras en el audio
    f'La frecuencia de muestreo es {sr} Hz',
    f'y(t) tiene {dt:.1f}s '#Duración
    , sep='\n')


y[t] tiene 441000 muestras
La frecuencia de muestreo es 4000 Hz
y(t) tiene 110.2s 


### Separar el audio en ventanas (Ya no hay que ejecutarlo)

In [ ]:
# Calcular el número de muestras por ventana de n segundos

n = 4  # Duración de la ventana en segundos (le puse 4)
samples_per_window = n * sr

# Crear ventanas de 4 segundos
audio_windows = []
for i in range(0, len(y), samples_per_window):
    window = y[i:i + samples_per_window]
    audio_windows.append(window)

print(f"El audio se ha dividido en {len(audio_windows)} ventanas de aproximadamente 4 segundos cada una.")

El audio se ha dividido en 8 ventanas de aproximadamente 4 segundos cada una.


In [ ]:
# Escuchar el audio de 4 segundos (Ventana 1) original

print("Ventana 1:")
display(Audio(audio_windows[0], rate=sr))

Ventana 1:


### Lectura de .txt para el síntoma

In [ ]:
txt_dir = '/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/'
txt_data_list = []

for filename in os.listdir(txt_dir):
    if filename.endswith('.txt'):
        file_path = os.path.join(txt_dir, filename)
        txt_data_list.append(file_path)

print(f"Se hallaron {len(txt_data_list)} archivos .txt.")

Se hallaron 920 archivos .txt.


In [ ]:
# Obtener el nombre del audio que se escogió arriba
audio_basename = os.path.splitext(os.path.basename(audio_file_path))[0]

# Hallar el .txt con el mismo nombre
corresponding_txt_file = None
for txt_file_path in txt_data_list:
    txt_basename = os.path.splitext(os.path.basename(txt_file_path))[0]
    if txt_basename == audio_basename:
        corresponding_txt_file = txt_file_path
        break

print (corresponding_txt_file)

/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/159_1b1_Pr_sc_Meditron.txt


In [ ]:
# Leer .txt y procesar

n = 4
processed_windows_info = []

corresponding_txt_file = "/content/respiratory_sound_database/Respiratory_Sound_Database/audio_and_txt_files/113_1b1_Pr_sc_Litt3200.txt"

if corresponding_txt_file:
    print(f"Procesando {os.path.basename(corresponding_txt_file)}:")
    with open(corresponding_txt_file, 'r') as f:
        lines = f.readlines()
        txt_windows = 0
        inc = 0 # Si no hay más líneas y no alcanza 4 segundos
        label_symp = []
        while txt_windows < len(lines):
            current_lines = []
            total_duration = 0
            symp_labels = []
            lines_window = 0  # Contador de líneas por ventana

            # Procesar líneas para la ventana de n segundos
            for i in range(n):
                if txt_windows + i < len(lines):
                    line = lines[txt_windows + i].strip().split()
                    if len(line) >= 4: #Verificar que tenga 4 columnas
                        try:
                            start_time = float(line[0]) # Tiempo de inicio
                            end_time = float(line[1]) # Tiempo final
                            duration = end_time - start_time

                            # Si la duración de la línea es igual o menor a n, sumar la siguiente pero solo considerar los síntomas antes de superar el tiempo
                            if duration <= n:
                                if total_duration + duration <= n:
                                    current_lines.append(line)
                                    total_duration += duration
                                    symp_labels.append(line[-2:])
                                    lines_window += 1
                                else:
                                    break # Detenerse si se alcanza el límite
                            else: # Si el tiempo es mayor a n, procesar una única línea
                                current_lines.append(line)
                                total_duration = duration
                                symp_labels.append(line[-2:])
                                lines_window = 1 # única ventana
                                break # Pasar a la siguiente

                        except ValueError:
                            print(f"Saltando línea {txt_windows + i + 1}: no hay valor numérico")
                            continue # Saltar línea
                    else:
                        print(f"Saltando línea {txt_windows + i + 1}: menos de cuatro columnas")
                        continue # Saltar línea
                else:
                    if total_duration < n and len(current_lines) > 0:
                      inc += 1
                    break # No hay más líneas que revisar

            if not current_lines:
                txt_windows += 1
                continue # Pasar de línea

            # Determina el síntoma (para label)
            final_symp = ['0', '0']
            for symp in symp_labels:
                if symp[0] == '1' or final_symp[0] == '1':
                    final_symp[0] = '1'
                if symp[1] == '1' or final_symp[1] == '1':
                    final_symp[1] = '1'

            label = final_symp[0] + final_symp[1]
            label_symp.append(label)

            # Guardar la información para procesar luego los audios
            processed_windows_info.append([len(current_lines), total_duration, label])

            # Mover con el número de líneas leídas
            txt_windows += lines_window

            print(f"Ventana procesada con {len(current_lines)} líneas. Duración total: {total_duration:.2f}s. Etiqueta: {label}")

else:
    print("No se encontró el archivo .txt correspondiente.")

Procesando 113_1b1_Pr_sc_Litt3200.txt:
Ventana procesada con 2 líneas. Duración total: 3.42s. Etiqueta: 00
Ventana procesada con 1 líneas. Duración total: 2.02s. Etiqueta: 00
Ventana procesada con 1 líneas. Duración total: 2.15s. Etiqueta: 00
Ventana procesada con 2 líneas. Duración total: 3.25s. Etiqueta: 00
Ventana procesada con 3 líneas. Duración total: 5.75s. Etiqueta: 00
Ventana procesada con 2 líneas. Duración total: 5.75s. Etiqueta: 00
Ventana procesada con 1 líneas. Duración total: 5.75s. Etiqueta: 00


In [ ]:
print(processed_windows_info)

[[2, 3.4177, '00'], [1, 2.0167, '00'], [1, 2.1511999999999993, '00'], [2, 3.254900000000001, '00'], [3, 5.753, '00'], [2, 5.753, '00'], [1, 5.753, '00']]


### Procesamiento de audio

In [ ]:
# Extraer segmentos de audio con la información procesada

audio_segments = []
duration_segments = []
current_sample = 0

for window_info in processed_windows_info:
    duration = window_info[1]  # Obtener la duración de la ventana (columna 2 (1))
    num_samples = int(duration * sr)

    # Extraer el segmento de la duración definida
    segment = y[current_sample : current_sample + num_samples]
    audio_segments.append(segment)
    duration_segments.append(duration)

    # Atualizar la posición final de la ventana para la siguiente iteración
    current_sample += num_samples

print(f"Se extrajeron {len(audio_segments)} segmentos de audio.")

Se extrajeron 7 segmentos de audio.


In [ ]:
print(duration_segments)

[3.4177, 2.0167, 2.1511999999999993, 3.254900000000001, 5.753, 5.753, 5.753]


In [ ]:
# Escuchar el segmento para verificar duración
if audio_segments:
    print("Primer segmento de audio:")
    display(Audio(audio_segments[0], rate=sr))
else:
    print("No hay segmentos de audio para reproducir.")

Primer segmento de audio:


In [ ]:
target_duration = n
adjusted_audio_segments = []

def search_healthy_audio():
  for i,j in enumerate(processed_windows_info):
    if j[2]=="00":
      return audio_segments[i]

try:
    #y_healthy, sr_healthy = librosa.load("/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/101_1b1_Al_sc_Meditron.wav", sr=sr)
    y_healthy = search_healthy_audio() # y_healthy debe ser numpy
    print("Audio healthy cargado correctamente.")
except Exception as e:
    print(f"Error loading healthy audio: {e}")
    y_healthy = None


for i, segment in enumerate(audio_segments):
    current_duration = len(segment) / sr
    target_samples = int(target_duration * sr)

    if current_duration < target_duration:
        if y_healthy is not None:
            # Calcular cuántas samples son necesarias
            needed_samples = target_samples - len(segment)

            # Obtener la mitad para hacer el padding al inicio y al final
            samples_before = needed_samples // 2
            samples_after = needed_samples - samples_before # Asegurar que todas las samples necesarias sean añadidas


            # Hacer append del audio, medio al inicio y medio al final
            adjusted_segment = np.concatenate((y_healthy[-samples_before:], segment, y_healthy[:samples_after]))

    elif current_duration > target_duration:
        # Truncar a n segundos si es mayor
        adjusted_segment = segment[:target_samples]
    else:
        # Si el segmento fuera exactamente de 4 segundos
        adjusted_segment = segment

    # Asegurar que el segmento ajustado es exactamente igual que target_samples
    if len(adjusted_segment) != target_samples:
        print(f"Warning: Segment {i} length is {len(adjusted_segment)} instead of {target_samples}. Truncating or padding with zeros.")
        if len(adjusted_segment) > target_samples:
            adjusted_segment = adjusted_segment[:target_samples]
        else:
            adjusted_segment = np.pad(adjusted_segment, (0, target_samples - len(adjusted_segment)), 'constant')
    adjusted_audio_segments.append(adjusted_segment)

print(f"Se ajustaron {len(adjusted_audio_segments)} segmentos de audio a {target_duration} segundos.")

Audio healthy cargado correctamente.
Se ajustaron 7 segmentos de audio a 4 segundos.


In [ ]:
print(len(adjusted_segment) != target_samples)

False


In [ ]:
# Escuchar el segmento para verificar duración
if audio_segments:
    print("Primer segmento de audio:")
    display(Audio(adjusted_audio_segments[0], rate=sr))
else:
    print("No hay segmentos de audio para reproducir.")

Primer segmento de audio:


In [ ]:
# Leer la duración de los audios
durations = [len(segment) / sr for segment in adjusted_audio_segments]

print("Duración de cada segmento de audio ajustado:")
for i, durationP in enumerate(durations):
    print(f"Segmento {i+1}: {durationP:.2f} segundos")

Duración de cada segmento de audio ajustado:
Segmento 1: 4.00 segundos
Segmento 2: 4.00 segundos
Segmento 3: 4.00 segundos
Segmento 4: 4.00 segundos
Segmento 5: 4.00 segundos
Segmento 6: 4.00 segundos
Segmento 7: 4.00 segundos
Segmento 8: 4.00 segundos


In [ ]:
labeled_audio_windows = []

# Ahora se usan las etiquetas directamente de processed_windows_info

if len(adjusted_audio_segments) == len(processed_windows_info):
    # Crear una lista de tuplas (segmento de audio ajustado, etiqueta)
    for i, segment in enumerate(adjusted_audio_segments):
        # Obtener la etiqueta de la ventana correspondiente en processed_windows_info
        label = processed_windows_info[i][2] # La etiqueta es el tercer elemento (índice 2)
        labeled_audio_windows.append((segment, label))

    print(f"Se han creado {len(labeled_audio_windows)} ventanas de audio con labels.")
    if labeled_audio_windows:
        # Mostrar la forma del segmento de audio y la etiqueta para la primera ventana
        print(f"Primer ventana etiquetada: Forma del segmento de audio: {labeled_audio_windows[0][0].shape}, Etiqueta: {labeled_audio_windows[0][1]}")
    else:
        print("No se crearon ventanas de audio etiquetadas.")
else:
    print(f"Error: El número de segmentos de audio ajustados ({len(adjusted_audio_segments)}) no coincide con el número de ventanas procesadas ({len(processed_windows_info)}). No se pueden crear ventanas etiquetadas.")

Se han creado 8 ventanas de audio con labels.
Primer ventana etiquetada: Forma del segmento de audio: (88200,), Etiqueta: 00


### Conversión a tensor

In [ ]:
# Esta conversión a tensor me toca revisarla mejor

import torch
import numpy as np

# Separar los datos de audio y las etiquetas

audio_data_list = [item[0] for item in labeled_audio_windows]
labels = label_symp

# Convertir la lista de arrays de numpy a una lista de tensores de torch
audio_tensors = [torch.from_numpy(audio).float() for audio in audio_data_list]

# Apilar los tensores para crear un único tensor, asumiendo que todos los tensores tienen el mismo tamaño después del padding
try:
    audio_tensor = torch.stack(audio_tensors)
    print("Datos de audio convertidos a tensor.")
    print(f"Forma del tensor de audio: {audio_tensor.shape}")
except RuntimeError as e:
    print(f"Error al apilar tensores: {e}")

# Convertir etiquetas a un tensor

unique_labels = list(set(labels))
label_mapping = {label: i for i, label in enumerate(unique_labels)}
encoded_labels = [label_mapping[label] for label in labels]
label_tensor = torch.tensor(encoded_labels)

print(f"Etiquetas (primeras 5): {labels[:5]}")
print(f"Etiquetas codificadas (primeras 5): {encoded_labels[:5]}")
print(f"Forma del tensor de etiquetas: {label_tensor.shape}")

Datos de audio convertidos a tensor.
Forma del tensor de audio: torch.Size([8, 88200])
Etiquetas (primeras 5): ['00', '00', '00', '00', '00']
Etiquetas codificadas (primeras 5): [0, 0, 0, 0, 0]
Forma del tensor de etiquetas: torch.Size([8])


### Filtro pasabanda

In [ ]:
from scipy.signal import butter, filtfilt

# Definir los parámetros
lowcut = 45.0  # Lower cutoff (Hz)
highcut = 55.0 # Upper cutoff (Hz)
order = 5      # Orden del filtro (no estoy seguro de cuánto debería ser)

# Implementación del filtro
nyquist = 0.5 * sr
low = lowcut / nyquist
high = highcut / nyquist
b, a = butter(order, [low, high], btype='band')

# Aplicar el filtro Butterworth
filtered_y = filtfilt(b, a, y)

print("Filtro pasabanda aplicado.")

Filtro pasabanda aplicado.


In [ ]:
# Calcular el número de muestras por ventana de n segundos

n = 4  # Duración de la ventana en segundos (le puse 4)
samples_per_window = n * sr

# Crear ventanas de 4 segundos a partir del audio filtrado
audio_windowsF = []
for i in range(0, len(filtered_y), samples_per_window):
    windowF = filtered_y[i:i + samples_per_window]
    audio_windowsF.append(windowF)

print(f"El audio filtrado se ha dividido en {len(audio_windows)} ventanas de aproximadamente 4 segundos cada una.")

El audio filtrado se ha dividido en 5 ventanas de aproximadamente 4 segundos cada una.


In [ ]:
# Escuchar el audio de 4 segundos (Ventana 1) filtrado

print("Ventana 1:")
display(Audio(audio_windowsF[0], rate=sr))

Ventana 1:


/usr/local/lib/python3.12/dist-packages/IPython/lib/display.py:175: RuntimeWarning: invalid value encountered in cast
  return scaled.astype("<h").tobytes(), nchan


In [ ]:
# Descargar

import soundfile as sf
from google.colab import files

output_filename_filtered = 'primera_ventana_filtrada.wav'
sf.write(output_filename_filtered, audio_windowsF[0], sr)

files.download(output_filename_filtered)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Procedimiento completo para todo el dataset

In [ ]:
# Lectura dataset completo

dataset_dir = '/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/'

audio_data_list = [] # Lista de los audios .wav
txt_data_list = [] # Lista de los textos .txt

for filename in os.listdir(dataset_dir):
    file_path = os.path.join(dataset_dir, filename)
    if os.path.isfile(file_path):
        if filename.endswith('.wav'):
            audio_data_list.append(file_path)
        elif filename.endswith('.txt'):
            txt_data_list.append(file_path)

print(f"Carga de {len(audio_data_list)} archivos de audio .wav.")
print(f"Carga de {len(txt_data_list)} archivos de texto .txt.")

Carga de 920 archivos de audio .wav.
Carga de 920 archivos de texto .txt.


In [ ]:
input_files = []

# Crear un diccionario para verificar que el archivo .txt concuerde con el .wav
txt_files = {os.path.splitext(os.path.basename(f))[0]: f for f in txt_data_list}

for audio_file_path in audio_data_list:
    audio_basename = os.path.splitext(os.path.basename(audio_file_path))[0]
    corresponding_txt_file = txt_files.get(audio_basename)

    if corresponding_txt_file:
        input_files.append((audio_file_path, corresponding_txt_file))

print(f"Diccionario creado con {len(input_files)} archivos audio - texto.")

Diccionario creado con 920 archivos audio - texto.


In [ ]:
all_processed_windows_info = []
n = 4 # Duración de cada ventana (4 segundos)

for audio_file_path, txt_file_path in input_files:
    processed_windows_info = [] # Información de cada ventana
    print(f"Procesando {os.path.basename(txt_file_path)}:")
    with open(txt_file_path, 'r') as f:
        lines = f.readlines()
        txt_windows = 0

        while txt_windows < len(lines):
            current_lines = []
            total_duration = 0
            symp_labels = []
            lines_window = 0

            for i in range(5): # Verificar hasta 5 líneas hasta formar una ventana de 4 segundos
                if txt_windows + i < len(lines):
                    line = lines[txt_windows + i].strip().split()
                    if len(line) >= 4: # Verificar que hayan 4 columnas
                        try:
                            start_time = float(line[0]) # Tiempo inicio ventana
                            end_time = float(line[1]) # Tiempo final ventana
                            duration = end_time - start_time

                            if total_duration + duration <= n: # Si la duración original no alcanza n segundos, se suma la siguiente
                                current_lines.append(line)
                                total_duration += duration
                                symp_labels.append(line[-2:]) # Solo se lee la línea hasta antes de superar n segundos
                                lines_window += 1
                            else:
                                break # Detener si al añadir esta se superaron los n segundos

                        except ValueError:
                            print(f"Saltando línea {txt_windows + i + 1} en {os.path.basename(txt_file_path)}: no hay valor numérico")
                            continue # Saltar línea
                    else:
                        print(f"Saltando línea {txt_windows + i + 1} en {os.path.basename(txt_file_path)}: menos de cuatro columnas")
                        continue # Saltar línea

                else:
                    break # No más líneas por leer

            if not current_lines:
                txt_windows += 1 # Pasar de línea
                continue

            # Dterminar el síntoma final con las condiciones dadas
            final_symp = ['0', '0']
            for symp in symp_labels:
                if symp[0] == '1' or final_symp[0] == '1':
                    final_symp[0] = '1'
                if symp[1] == '1' or final_symp[1] == '1':
                    final_symp[1] = '1'

            label = final_symp[0] + final_symp[1] # Label

            processed_windows_info.append([len(current_lines), total_duration, label]) # Guardar información de la ventana

            txt_windows += lines_window

    all_processed_windows_info.append((audio_file_path, processed_windows_info)) # Guardar información de todas las ventanas con su audio
    print(f"Se procesaron {len(processed_windows_info)} ventanas para {os.path.basename(txt_file_path)}")

print(f"Final del proceso: {len(all_processed_windows_info)} archivos .txt analizados.")

Procesando 145_2b2_Lr_mc_AKGC417L.txt:
Se procesaron 5 ventanas para 145_2b2_Lr_mc_AKGC417L.txt
Procesando 133_2p4_Pl_mc_AKGC417L.txt:
Se procesaron 7 ventanas para 133_2p4_Pl_mc_AKGC417L.txt
Procesando 158_1p2_Tc_mc_AKGC417L.txt:
Se procesaron 7 ventanas para 158_1p2_Tc_mc_AKGC417L.txt
Procesando 156_8b3_Lr_mc_AKGC417L.txt:
Se procesaron 8 ventanas para 156_8b3_Lr_mc_AKGC417L.txt
Procesando 154_2b4_Tc_mc_AKGC417L.txt:
Se procesaron 7 ventanas para 154_2b4_Tc_mc_AKGC417L.txt
Procesando 133_3p4_Tc_mc_AKGC417L.txt:
Se procesaron 7 ventanas para 133_3p4_Tc_mc_AKGC417L.txt
Procesando 122_2b3_Tc_mc_LittC2SE.txt:
Se procesaron 2 ventanas para 122_2b3_Tc_mc_LittC2SE.txt
Procesando 160_1b4_Pl_mc_AKGC417L.txt:
Se procesaron 0 ventanas para 160_1b4_Pl_mc_AKGC417L.txt
Procesando 193_1b4_Lr_mc_AKGC417L.txt:
Se procesaron 8 ventanas para 193_1b4_Lr_mc_AKGC417L.txt
Procesando 177_1b4_Pr_mc_AKGC417L.txt:
Se procesaron 3 ventanas para 177_1b4_Pr_mc_AKGC417L.txt
Procesando 205_1b3_Ll_mc_AKGC417L.txt:
S

In [ ]:
print (len(all_processed_windows_info))
print (all_processed_windows_info[919])

920
('/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/104_1b1_Ar_sc_Litt3200.wav', [[2, 2.9628, '01'], [1, 2.1457, '01'], [1, 2.1087, '01'], [2, 3.4578000000000007, '01'], [2, 3.7059999999999995, '01'], [1, 2.154, '01'], [1, 2.512999999999998, '01'], [2, 3.9030000000000022, '00'], [2, 2.632999999999999, '00']])


In [45]:
n = 4  # Duración de ventana por si no se ha definido

all_processed_data = [] # Lista de tuplas para toda la información de procesamiento (adjusted_segment, label, original_audio_path)

# Cargar audios predeterminados para cada estetoscopio por si no hay healthy o es muy corto
try:
    y_healthy_Med, sr_healthy_Med = librosa.load("/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/101_1b1_Al_sc_Meditron.wav")
    y_healthy_Lit1, sr_healthy_Lit1 = librosa.load("/content/respiratory_sound_database/Respiratory_Sound_Database/audio_and_txt_files/124_1b1_Pr_sc_Litt3200.wav")
    y_healthy_Lit2, sr_healthy_Lit2 = librosa.load("/content/respiratory_sound_database/Respiratory_Sound_Database/audio_and_txt_files/135_2b3_Al_mc_LittC2SE.wav")
    y_healthy_AKG, sr_healthy_AKG = librosa.load("/content/respiratory_sound_database/Respiratory_Sound_Database/audio_and_txt_files/107_2b5_Tc_mc_AKGC417L.wav")
    print("Audio cargado correctamente.")
except Exception as e:
    print(f"Error cargando backup: {e}")
    y_healthy_backup = None


for audio_file_path, processed_windows_info in all_processed_windows_info:
    try:
        y, current_sr = librosa.load(audio_file_path) # Cargar el audio actual, obtener sr
        target_samples = int(n * current_sr)

        # Hallar un segmento healthy para concatenar
        y_healthy_current_file = None

        for window_info in processed_windows_info:
            if window_info[2] == "00": # Posición 2 de la tupla es síntoma
                # Calcular tiempos de inicio y dalisa
                start_time = float(window_info[0]) # Tiempo inicio ventana
                duration = window_info[1] # Tiempo final ventana
                end_time = start_time + duration  # Total tiempo por si cualquier cosa
                start_sample = int(start_time * current_sr) # Inicio sample
                end_sample = int(end_time * current_sr) # Final sample

                # Extraer el segmento requerido para healthy
                healthy_segment_candidate = y[start_sample:end_sample]

                # Verificar si alcanza para completar n segundos. Le puse 1 segundo por la distribución de los audios
                if len(healthy_segment_candidate) / current_sr >= 1:
                     y_healthy_current_file = healthy_segment_candidate
                     print(f"Segmento añadido desde {os.path.basename(audio_file_path)}.")
                     break # Usar el segmento healthy elegido

        current_sample = 0 # Inicializar current_sample

        for window_info in processed_windows_info:
            duration = window_info[1]  # Obtener la duración de la ventana
            num_samples = int(duration * current_sr)

            # Extraer el segmento
            segment = y[current_sample : current_sample + num_samples]

            # Ajuste de duración de la ventana
            current_duration = len(segment) / current_sr

            if current_duration < n:  # Si la duración es menor a n segundos añadir
                needed_samples = target_samples - len(segment)
                # Verificar que y_healthy_current_file no sea None y tenga samples
                if y_healthy_current_file is not None and len(y_healthy_current_file) >= needed_samples:

                    samples_before = needed_samples // 2 # Samples/2 al inicio
                    samples_after = needed_samples - samples_before # Samples/2 al final

                    # Concatenar al segmento original usando y_healthy_current_file
                    if samples_before > len(y_healthy_current_file) // 2 or samples_after > len(y_healthy_current_file) // 2: #Verificar que no haya error, no debe salir
                         print(f"Error: El segmento en {os.path.basename(audio_file_path)} no es suficiente. No completado.")
                    else:
                         # Tomar las samples y hacer la concatenación
                         healthy_midpoint = len(y_healthy_current_file) // 2
                         adjusted_segment = np.concatenate((y_healthy_current_file[healthy_midpoint - samples_before : healthy_midpoint],
                                                             segment,
                                                             y_healthy_current_file[healthy_midpoint : healthy_midpoint + samples_after]))

                # Check if any of the healthy backup audios are not None and have enough samples
                elif (y_healthy_Med is not None and len(y_healthy_Med) >= needed_samples) or \
                     (y_healthy_Lit1 is not None and len(y_healthy_Lit1) >= needed_samples) or \
                     (y_healthy_Lit2 is not None and len(y_healthy_Lit2) >= needed_samples) or \
                     (y_healthy_AKG is not None and len(y_healthy_AKG) >= needed_samples): # Verificar sobre los healthy backup
                     print(f"Usando audio de backup para {os.path.basename(audio_file_path)}.")
                     samples_before = needed_samples // 2
                     samples_after = needed_samples - samples_before

                     # Tomar las muestras del audio backup predefinido para cada estetoscopio si no hay healthy o no alcanza 4 segundos
                     if "Meditron" in audio_file_path and y_healthy_Med is not None:
                         y_healthy_backup = y_healthy_Med
                     elif "Litt3200" in audio_file_path and y_healthy_Lit1 is not None:
                         y_healthy_backup = y_healthy_Lit1
                     elif "LittC2SE" in audio_file_path and y_healthy_Lit2 is not None:
                         y_healthy_backup = y_healthy_Lit2
                     elif "AKGC" in audio_file_path and y_healthy_AKG is not None:
                         y_healthy_backup = y_healthy_AKG
                     else:
                          print(f"Error: No hay audio de backup.")
                          y_healthy_backup = None # Reiniciar backup para no usarlo si no existe


                     if y_healthy_backup is not None:
                        healthy_midpoint_backup = len(y_healthy_backup) // 2
                        # Asegurar no pedir más muestras de las disponibles
                        if samples_before > len(y_healthy_backup) // 2 or samples_after > len(y_healthy_backup) // 2:
                             print(f"Error: El archivo de backup healthy para {os.path.basename(audio_file_path)} no alcanza. No completado.")
                        else:
                             adjusted_segment = np.concatenate((y_healthy_backup[healthy_midpoint_backup - samples_before : healthy_midpoint_backup],
                                                                 segment,
                                                                 y_healthy_backup[healthy_midpoint_backup : healthy_midpoint_backup + samples_after]))


                else:
                    print(f"Error: El segmento en {os.path.basename(audio_file_path)} no es suficiente. No completado.")

            elif current_duration > n:
                # Truncar si se pasa de n segundos
                adjusted_segment = segment[:target_samples]
            else:
                # Si todo es perfecto
                adjusted_segment = segment

            # Verificar que el número de samples sea el correcto para poder recuperar después el audio
            if len(adjusted_segment) != target_samples:
                print(f"Error en {os.path.basename(audio_file_path)}. Esperando {target_samples} muestras, se tienen {len(adjusted_segment)}. No completado.")


            all_processed_data.append((adjusted_segment, window_info[2], audio_file_path)) # Guardar segmento, label, y path del archivo para leerlo después

            # Actualizar current_sample con el largo original
            current_sample += num_samples

    except Exception as e:
        print(f"Error procesando {audio_file_path}: {e}")


print(f"Fin del procesamiento. Se crearon {len(all_processed_data)} segmentos procesados.")

Audio cargado correctamente.
Segmento añadido desde 145_2b2_Lr_mc_AKGC417L.wav.
Usando audio de backup para 133_2p4_Pl_mc_AKGC417L.wav.
Usando audio de backup para 133_2p4_Pl_mc_AKGC417L.wav.
Usando audio de backup para 133_2p4_Pl_mc_AKGC417L.wav.
Usando audio de backup para 133_2p4_Pl_mc_AKGC417L.wav.
Usando audio de backup para 133_2p4_Pl_mc_AKGC417L.wav.
Usando audio de backup para 133_2p4_Pl_mc_AKGC417L.wav.
Usando audio de backup para 133_2p4_Pl_mc_AKGC417L.wav.
Segmento añadido desde 158_1p2_Tc_mc_AKGC417L.wav.
Segmento añadido desde 156_8b3_Lr_mc_AKGC417L.wav.
Segmento añadido desde 154_2b4_Tc_mc_AKGC417L.wav.
Segmento añadido desde 133_3p4_Tc_mc_AKGC417L.wav.
Segmento añadido desde 122_2b3_Tc_mc_LittC2SE.wav.
Usando audio de backup para 193_1b4_Lr_mc_AKGC417L.wav.
Usando audio de backup para 193_1b4_Lr_mc_AKGC417L.wav.
Usando audio de backup para 193_1b4_Lr_mc_AKGC417L.wav.
Usando audio de backup para 193_1b4_Lr_mc_AKGC417L.wav.
Usando audio de backup para 193_1b4_Lr_mc_AKGC417

In [46]:
# En all_processed_data se tiene una tupla de tres posiciones: (Audio, Label, Path)
# Cada tupla es un segmento, el tercer elemento es para saber a qué audio corresponde

print(all_processed_data[2])

(array([ 0.07892759,  0.07781386,  0.07812082, ..., -0.01763213,
       -0.01772021, -0.01791468], dtype=float32), '00', '/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/145_2b2_Lr_mc_AKGC417L.wav')


In [47]:
all_processed_data[2][2]

'/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/145_2b2_Lr_mc_AKGC417L.wav'

In [48]:
# Escuchar el segmento para verificar duración

y, sr_prueba = librosa.load(all_processed_data[2][2]) # Cargar el audio actual, obtener sr

print (current_sr, sr_prueba)

22050 22050


In [56]:
if all_processed_data:
    print("Segmento de audio:")
    display(Audio(all_processed_data[2][0], rate=sr_prueba))
else:
    print("No hay segmentos de audio para reproducir.")

Segmento de audio:


In [55]:
# Leer la duración de cualquier segmento procesado
if all_processed_data:
    p_segment = all_processed_data[2][0]  # Obtener la información de la tupla en posición 0 (audio)
    duration_segment = len(p_segment) / sr_prueba # Usar el sample de prueba porque puede cambiar

    print(f"La duración del segmento es: {duration_segment:.2f} segundos")
else:
    print("No hay segmentos para leer.")

La duración del segmento es: 4.00 segundos


In [53]:
import torch
import numpy as np

# Separar los datos de audio y los labels
audio_data_list = [item[0] for item in all_processed_data]
labels = [item[1] for item in all_processed_data]

# Convertir la lista de numpy a lista de Torch tensor
audio_tensors = [torch.from_numpy(audio).float() for audio in audio_data_list]

# Sobreponer tensores
try:
    audio_tensor = torch.stack(audio_tensors)
    print("Covertido a tensor.")
    print(f"Forma del tensor de audio: {audio_tensor.shape}")
except RuntimeError as e:
    print(f"Error: {e}")

# Convertir labels a tensor (Asumiéndolos como string)

unique_labels = sorted(list(set(labels))) # Sort unique labels for consistent mapping
label_mapping = {label: i for i, label in enumerate(unique_labels)}
encoded_labels = [label_mapping[label] for label in labels]
label_tensor = torch.tensor(encoded_labels)

print(f"Labels originales (primeros 5): {labels[:5]}")
print(f"Labels codificados (primeros 5): {encoded_labels[:5]}")
print(f"Forma del tensor de labels: {label_tensor.shape}")

Covertido a tensor.
Forma del tensor de audio: torch.Size([5189, 88200])
Labels originales (primeros 5): ['00', '00', '00', '00', '00']
Labels codificados (primeros 5): [0, 0, 0, 0, 0]
Forma del tensor de labels: torch.Size([5189])
